# 1.0 Матчи по 1 / 2 / 3 фильтрам (walk-forward)

Считает пороги в 6 окнах и отбирает ставки текущего сезона Y по порогам окна, заканчивающегося годом Y-1 (без утечки).

**Фильтры**
- F1: `k2 <= Threshold_k2` (шаг 2)
- F2: F1-пул + второй критерий из колонок F-P (шаг 3, если матчей > 800) либо пара `k2 <= t` + колонка F-P на всём пуле (шаг 4, если матчей <= 800)
- F3: F2 + порог `THIRD_COL THIRD_OP x`, подобранный на обучающем окне (шаг 5; по умолчанию `Surf p <`)

Параметры задаются в первой ячейке.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load(open(ROOT / "data" / "config.yaml", encoding="utf-8"))
F_TO_P = CFG["threshold_search"]["criteria_columns_F_to_P"]
MIN_K2 = CFG["threshold_search"]["min_matches_under_threshold"]
SPLIT = CFG["threshold_search"]["matches_split"]

PROFILE = "clay"        # "clay" или "hard_grass"
FILTER_LEVEL = 3        # 1, 2 или 3
THIRD_COL, THIRD_OP = "Surf p", "<"
SRC = ROOT / "data" / "raw" / "CRITERIA_DATA.xlsx"
OUT = ROOT / "data" / "processed" / f"{PROFILE.upper()}_{FILTER_LEVEL}filters.xlsx"
GROUPS = {g["name"]: g["sheets"] for g in CFG["groups"][PROFILE]}
WINDOWS = CFG["threshold_search"]["windows"]
SHEET2GROUP = {s: g for g, ss in GROUPS.items() for s in ss}

In [ ]:
def load(path, sheets):
    d = pd.read_excel(path, sheet_name=list(sheets), usecols=range(25))
    df = pd.concat([v.assign(sheet=k, group=SHEET2GROUP[k]) for k, v in d.items()], ignore_index=True)
    df = df.dropna(subset=["k2", "win", "Date"]).reset_index(drop=True)
    df["year"] = pd.to_datetime(df["Date"]).dt.year
    df["win2"] = np.where(df["win"] == 0, 100.0, -100.0 / (df["k2"] - 1))
    return df

df = load(SRC, SHEET2GROUP)
COLS = [c for c in df.columns if c in F_TO_P]
len(df), df.groupby("group").size().to_dict()

In [ ]:
def k2_threshold(w):
    a = w.groupby("k2")["win2"].agg(["sum", "size"]).sort_index()
    cum, n = a["sum"].cumsum(), a["size"].cumsum()
    cum = cum[n >= MIN_K2]
    t = cum.idxmax()
    return t, cum.max(), int(n[t])

def best_single(w, cols, min_n, ops=("<", ">")):
    """Лучший порог по одной колонке (макс. сумма win2, не меньше min_n матчей)."""
    best = None
    for col in cols:
        x = w[col].values; ok = ~np.isnan(x)
        xs, vs = x[ok], w["win2"].values[ok]
        o = np.argsort(xs, kind="stable"); xs, vs = xs[o], vs[o]
        cs = np.concatenate([[0], np.cumsum(vs)]); N = len(xs)
        u, first = np.unique(xs, return_index=True); last = np.append(first[1:], N)
        for op in ops:
            n, s = (first, cs[first]) if op == "<" else (N - last, cs[N] - cs[last])
            m = n >= min_n
            if m.any():
                i = np.argmax(np.where(m, s, -1e18))
                if best is None or s[i] > best[0]:
                    best = (s[i], col, op, u[i], int(n[i]))
    return best

def crit_mask(d, col, op, thr):
    return d[col] > thr if op == ">" else d[col] < thr

def fit_window(w):
    """Шаги 2-5 на обучающем окне w одной группы."""
    t, mx, n = k2_threshold(w)
    r = dict(Threshold_k2=t, Max_cum_win2=mx, Matches_under_threshold=n, Total_matches=len(w))
    if n > SPLIT:                                   # шаг 3
        b = best_single(w[w["k2"] <= t], COLS, MIN_K2); r["mode"] = ">800"; k2_lim = t
    else:                                           # шаг 4: k2<=t2 + колонка на всём пуле
        best = None
        for t2 in np.sort(w["k2"].unique()):
            sub = w[w["k2"] <= t2]
            if len(sub) < MIN_K2: continue
            b2 = best_single(sub, COLS, MIN_K2)
            if b2 and (best is None or b2[0] > best[0][0]): best = (b2, t2)
        b, k2_lim = best; r["mode"] = "<=800"
    r.update(k2_limit=k2_lim, crit2_col=b[1], crit2_op=b[2], crit2_value=b[3], Sum_after_2=b[0])
    f2 = w[(w["k2"] <= k2_lim) & crit_mask(w, b[1], b[2], b[3])]
    r["Matches_after_2"] = len(f2)
    t3 = best_single(f2, [THIRD_COL], MIN_K2, ops=(THIRD_OP,))
    r.update(third_value=t3[3] if t3 else np.nan, Sum_after_3=t3[0] if t3 else f2["win2"].sum(),
             Matches_after_3=t3[4] if t3 else len(f2))
    return r

def apply_filters(d, r, level):
    m = d["k2"] <= (r["Threshold_k2"] if level == 1 else r["k2_limit"])
    if level >= 2: m &= crit_mask(d, r["crit2_col"], r["crit2_op"], r["crit2_value"])
    if level >= 3 and not np.isnan(r["third_value"]): m &= crit_mask(d, THIRD_COL, THIRD_OP, r["third_value"])
    return d[m]

In [ ]:
rows = []
for w_ in WINDOWS:
    for g in GROUPS:
        sub = df[(df.group == g) & (df.year <= w_["end_year"]) & (df.year >= (w_.get("start_year") or 0))]
        rows.append(dict(Window=w_["label"], window_end_year=w_["end_year"], Group=g, **fit_window(sub)))
TH = pd.DataFrame(rows)
TH.round(4)

In [ ]:
# Ставки сезона Y: пороги строго из окна, заканчивающегося Y-1
W0 = {w_["end_year"]: (w_.get("start_year") or 0) for w_ in WINDOWS}
bets, res = [], []
for _, r in TH[TH.window_end_year < 2025].iterrows():
    Y = r.window_end_year + 1
    for g_sheet in GROUPS[r.Group]:
        d = df[df.sheet == g_sheet]
        prev = apply_filters(d[(d.year <= r.window_end_year) & (d.year >= W0[r.window_end_year])], r, FILTER_LEVEL)
        cur = apply_filters(d[d.year == Y], r, FILTER_LEVEL)
        res.append({"tourn_sheet": g_sheet, "group": r.Group, "prev.seasons": prev.win2.sum(), "curr.season": cur.win2.sum(),
                    "year": Y, "threshold": r.k2_limit, "prev_count": len(prev), "curr_count": len(cur)})
        bets.append(cur.assign(eval_year=Y, window=r.Window, threshold_k2=r.k2_limit))
RES = pd.DataFrame(res).sort_values("prev.seasons").reset_index(drop=True)
BETS = pd.concat(bets).drop(columns="year").sort_values(["eval_year", "Date"])
print(len(BETS), round(BETS.win2.sum(), 1))
RES.groupby("year")[["curr.season", "curr_count"]].sum()

In [ ]:
with pd.ExcelWriter(OUT) as xw:
    TH.to_excel(xw, sheet_name="Thresholds", index=False)
    RES.to_excel(xw, sheet_name="Results_by_tourn_sheet", index=False)
    BETS.to_excel(xw, sheet_name="curr_games", index=False)
OUT